# Plant Disease Detection -- Colab Training (EfficientNet-B0)

Colab/GPU port of `model/train.py`. Same model logic (stratified split, per-class
augmentation boost, class-weighted loss, EfficientNet-B0 transfer learning,
per-class precision/recall/F1) -- only the environment/infra parts differ:
GPU check instead of blind reinstall, Google Drive for persistence, and a
Kaggle-based re-download of the raw datasets (re-uploading ~32.7k images from
a local machine would be far slower than re-downloading from Kaggle).

Run cells top to bottom. See the bottom of this notebook for a per-epoch,
per-class F1 summary table once training finishes (or partially finishes).

## 1. Check runtime (GPU)

Don't reinstall torch -- Colab ships with a CUDA-enabled build already. Just confirm it's actually being used.

In [ ]:
import torch

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected.")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU (T4), then re-run this cell.")


In [ ]:
# scikit-learn and the kaggle CLI are usually preinstalled on Colab; this is a
# no-op if so, and only actually installs anything if they're missing.
!pip install -q scikit-learn kaggle
print("Dependency check done.")


## 2. Mount Google Drive

Colab runtimes disconnect and wipe local disk between sessions, so the raw
datasets, `data/processed/`, and per-epoch checkpoints all live on Drive
instead -- they'll still be there next time you connect, even after a
disconnect.

**Expected Drive folder**: `MyDrive/Sasya/` (edit `DRIVE_ROOT` below if you'd
rather use a different folder name/location). Nothing needs to exist there
beforehand -- this cell creates the subfolders it needs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Sasya')  # <-- change this if your Drive folder differs

DATA_RAW = DRIVE_ROOT / 'data' / 'raw'
DATA_PROCESSED = DRIVE_ROOT / 'data' / 'processed'
MODEL_DIR = DRIVE_ROOT / 'model'
CHECKPOINT_DIR = MODEL_DIR / 'checkpoints'

for d in (DATA_RAW, DATA_PROCESSED, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Drive root:       ", DRIVE_ROOT)
print("Raw data dir:     ", DATA_RAW)
print("Processed data dir:", DATA_PROCESSED)
print("Checkpoint dir:   ", CHECKPOINT_DIR)


## 3. Kaggle authentication

Expects a `kaggle.json` API token (Kaggle account -> Settings -> API -> Create
New Token). Two ways this cell can get it:

- If `kaggle.json` already exists in your Drive at `Sasya/kaggle.json`, it's
  copied from there automatically -- no upload needed.
- Otherwise, you'll be prompted to upload it once. It's then saved to Drive
  so future sessions skip the upload step.

Your Kaggle credentials are only ever read locally in your own Colab runtime
and copied to your own Drive -- nothing is sent anywhere else by this notebook.

In [ ]:
import os
import shutil

KAGGLE_JSON_DRIVE_PATH = DRIVE_ROOT / 'kaggle.json'
kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_target = kaggle_dir / 'kaggle.json'

if KAGGLE_JSON_DRIVE_PATH.exists():
    shutil.copy(KAGGLE_JSON_DRIVE_PATH, kaggle_json_target)
    print(f"Copied kaggle.json from Drive: {KAGGLE_JSON_DRIVE_PATH}")
else:
    from google.colab import files
    print("kaggle.json not found in Drive -- please upload it now.")
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    Path(uploaded_name).rename(kaggle_json_target)
    shutil.copy(kaggle_json_target, KAGGLE_JSON_DRIVE_PATH)
    print(f"Saved a copy to Drive at {KAGGLE_JSON_DRIVE_PATH} for future sessions.")

os.chmod(kaggle_json_target, 0o600)
print("Kaggle auth ready.")


## 4. Download + unpack the raw datasets

Same three sources as `data/README.md`. **Check `KAGGLE_DATASETS["plantvillage"]`
before running** -- unlike the other two sources, PlantVillage doesn't have one
canonical Kaggle listing in our docs (`data/README.md` just says "search
PlantVillage Dataset on Kaggle"). The slug below is a commonly-used ~54k-image
mirror with the original `Tomato___Bacterial_spot`-style folder names (the
same naming `data/prepare_dataset.py`'s `CLASS_MAPPING` expects) -- confirm
it's the same one you used locally, or swap in the exact slug you know works.

Skips the download entirely if a source's raw folder on Drive already has
content, so re-running this notebook in a later session doesn't re-download
everything.

In [ ]:
KAGGLE_DATASETS = {
    # CONFIRM this matches what you downloaded locally -- see markdown above.
    "plantvillage": "abdallahalidev/plantvillage-dataset",
    "five-crop-diseases": "shubham2703/five-crop-diseases-dataset",
    "cotton-disease": "seroshkarim/cotton-leaf-disease-dataset",
}


def flatten_single_wrapper_dir(dest_dir):
    """Kaggle zips sometimes extract into one extra wrapper folder
    (e.g. dest_dir/PlantVillage/<classes>/ instead of dest_dir/<classes>/),
    occasionally alongside a stray top-level file (a metadata/description
    file bundled in the zip). Checking for exactly one entry total is too
    strict -- it misses the "one folder plus a stray file" case. Instead:
    if dest_dir has exactly one *subdirectory* (regardless of stray files),
    hoist that subdirectory's contents up and repeat, so any depth of
    single-folder wrapping gets flattened."""
    while True:
        entries = list(dest_dir.iterdir())
        subdirs = [e for e in entries if e.is_dir()]
        if len(subdirs) != 1:
            break
        wrapper = subdirs[0]
        for item in wrapper.iterdir():
            shutil.move(str(item), str(dest_dir / item.name))
        wrapper.rmdir()


def download_and_unzip(slug, dest_dir):
    if dest_dir.exists() and any(dest_dir.iterdir()):
        print(f"{dest_dir} already has content -- skipping download.")
        return

    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {slug} ...")
    !kaggle datasets download -d {slug} -p {dest_dir} --force

    zips = list(dest_dir.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"No zip file downloaded for {slug} into {dest_dir}")

    zip_path = zips[0]
    print(f"Unzipping {zip_path.name} ...")
    import zipfile
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest_dir)
    zip_path.unlink()

    flatten_single_wrapper_dir(dest_dir)


for source_key, slug in KAGGLE_DATASETS.items():
    print(f"\n=== {source_key} ({slug}) ===")
    download_and_unzip(slug, DATA_RAW / source_key)


### 4b. One-off repair for already-downloaded folders

Only needed if you already ran the download cell above with the *old*
flatten logic (strict "exactly one entry total" check), which missed cases
where a wrapper folder was sitting alongside a stray top-level file --
`plantvillage/` was left as `plantvillage/plantvillage dataset/<classes>/`
and `five-crop-diseases/` as `five-crop-diseases/Crop Diseases/<classes>/`,
instead of the classes sitting directly under each source root.

This cell re-applies the corrected `flatten_single_wrapper_dir` (defined in
the cell above -- run that one first if you haven't) directly to the raw
folders already on Drive, with **no re-download**. It's idempotent: if a
source is already flat, `flatten_single_wrapper_dir` sees more than one
subdirectory immediately and does nothing, so it's safe to run even if
you're not sure whether a given source needs it. Skip this cell entirely on
a fresh run where the download cell above already used the fixed logic.

In [ ]:
for source_key in KAGGLE_DATASETS:
    source_dir = DATA_RAW / source_key
    if not source_dir.is_dir():
        print(f"{source_dir} does not exist yet -- nothing to repair (run the download cell first).")
        continue
    before = sorted(p.name for p in source_dir.iterdir())
    flatten_single_wrapper_dir(source_dir)
    after = sorted(p.name for p in source_dir.iterdir())
    if before == after:
        print(f"{source_key}: already flat, no change ({len(after)} top-level entries).")
    else:
        print(f"{source_key}: flattened. Before: {before[:3]}{'...' if len(before) > 3 else ''} -> After: {len(after)} entries, e.g. {after[:3]}")


## 5. Merge into `data/processed/`

Ported directly from `data/prepare_dataset.py` -- same `CLASS_MAPPING`, same
fuzzy folder-name matching, same corrupted-image skipping. Only the paths
point at Drive instead of a local `data/raw/` and `data/processed/`. Keep
this in sync with `data/prepare_dataset.py` if that script changes.

Skips the merge entirely if `data/processed/` on Drive already has all 30
class folders populated, so re-running this notebook doesn't redo it.

In [ ]:
import sys
from PIL import Image

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Unified class name -> (source key, documented source folder name).
CLASS_MAPPING = {
    "Tomato_Bacterial_Spot": ("plantvillage", "Tomato___Bacterial_spot"),
    "Tomato_Early_Blight": ("plantvillage", "Tomato___Early_blight"),
    "Tomato_Late_Blight": ("plantvillage", "Tomato___Late_blight"),
    "Tomato_Leaf_Mold": ("plantvillage", "Tomato___Leaf_Mold"),
    "Tomato_Septoria_Leaf_Spot": ("plantvillage", "Tomato___Septoria_leaf_spot"),
    "Tomato_Spider_Mites": ("plantvillage", "Tomato___Spider_mites Two-spotted_spider_mite"),
    "Tomato_Target_Spot": ("plantvillage", "Tomato___Target_Spot"),
    "Tomato_Yellow_Leaf_Curl_Virus": ("plantvillage", "Tomato___Tomato_Yellow_Leaf_Curl_Virus"),
    "Tomato_Mosaic_Virus": ("plantvillage", "Tomato___Tomato_mosaic_virus"),
    "Tomato_Healthy": ("plantvillage", "Tomato___healthy"),
    "Chili_Bacterial_Spot": ("plantvillage", "Pepper,_bell___Bacterial_spot"),
    "Chili_Healthy": ("plantvillage", "Pepper,_bell___healthy"),
    "Potato_Early_Blight": ("plantvillage", "Potato___Early_blight"),
    "Potato_Late_Blight": ("plantvillage", "Potato___Late_blight"),
    "Potato_Healthy": ("plantvillage", "Potato___healthy"),
    "Corn_Common_Rust": ("plantvillage", "Corn_(maize)___Common_rust_"),
    "Corn_Gray_Leaf_Spot": ("plantvillage", "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot"),
    "Corn_Northern_Leaf_Blight": ("plantvillage", "Corn_(maize)___Northern_Leaf_Blight"),
    "Corn_Healthy": ("plantvillage", "Corn_(maize)___healthy"),
    "Rice_Brown_Spot": ("five-crop-diseases", "Rice___Brown_Spot"),
    "Rice_Leaf_Blast": ("five-crop-diseases", "Rice___Leaf_Blast"),
    "Rice_Neck_Blast": ("five-crop-diseases", "Rice___Neck_Blast"),
    "Rice_Healthy": ("five-crop-diseases", "Rice___Healthy"),
    "Sugarcane_Red_Rot": ("five-crop-diseases", "Sugarcane__Red_Rot"),
    "Sugarcane_Bacterial_Blight": ("five-crop-diseases", "Sugarcane__Bacterial Blight"),
    "Sugarcane_Healthy": ("five-crop-diseases", "Sugarcane__Healthy"),
    "Cotton_Bacterial_Blight": ("cotton-disease", "bacterial_blight"),
    "Cotton_Curl_Virus": ("cotton-disease", "curl_virus"),
    "Cotton_Fusarium_Wilt": ("cotton-disease", "fussarium_wilt"),  # source folder misspells "fusarium"
    "Cotton_Healthy": ("cotton-disease", "healthy"),
}

SOURCE_ROOTS = {key: DATA_RAW / key for key in {"plantvillage", "five-crop-diseases", "cotton-disease"}}


def normalize_folder_name(name):
    return "".join(ch.lower() for ch in name if ch.isalnum())


def find_source_folder(source_root, documented_name):
    if documented_name is None:
        return None
    exact_match = source_root / documented_name
    if exact_match.is_dir():
        return exact_match
    target = normalize_folder_name(documented_name)
    for entry in source_root.iterdir():
        if entry.is_dir() and normalize_folder_name(entry.name) == target:
            return entry
    return None


def is_image_file(path):
    return path.suffix.lower() in IMAGE_EXTENSIONS


def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def process_class(unified_class, source_key, documented_name):
    source_root = SOURCE_ROOTS[source_key]
    source_folder = find_source_folder(source_root, documented_name)

    if source_folder is None:
        available = sorted(p.name for p in source_root.iterdir() if p.is_dir()) if source_root.is_dir() else []
        print(f"  ERROR: could not find a folder matching '{documented_name}' under {source_root}.")
        print(f"    -> Folders actually present there: {available}")
        return None

    dest_dir = DATA_PROCESSED / unified_class
    dest_dir.mkdir(parents=True, exist_ok=True)
    copied = 0
    skipped = 0

    for entry in sorted(source_folder.iterdir()):
        if not entry.is_file() or not is_image_file(entry):
            skipped += 1 if entry.is_file() else 0
            continue
        if not is_valid_image(entry):
            skipped += 1
            continue
        shutil.copy2(entry, dest_dir / entry.name)
        copied += 1

    print(f"  copied {copied} image(s) from {source_folder.name}  ({skipped} skipped)")
    return copied


def processed_already_populated(processed_dir, expected_classes):
    if not processed_dir.exists():
        return False
    existing = {p.name for p in processed_dir.iterdir() if p.is_dir()}
    if not expected_classes.issubset(existing):
        return False
    sample_class = next(iter(expected_classes))
    return any((processed_dir / sample_class).iterdir())


expected_classes = set(CLASS_MAPPING.keys())

if processed_already_populated(DATA_PROCESSED, expected_classes):
    print(f"{DATA_PROCESSED} already has all {len(expected_classes)} classes populated -- skipping merge.")
else:
    had_errors = False
    for unified_class, (source_key, documented_name) in CLASS_MAPPING.items():
        print(f"[{unified_class}]")
        count = process_class(unified_class, source_key, documented_name)
        if count is None:
            had_errors = True
    if had_errors:
        print("\nOne or more classes had errors -- see ERROR lines above. Fix KAGGLE_DATASETS or the raw folders and re-run this cell.")
        sys.exit(1)
    print("\nMerge complete.")


## 6. Training setup

Ported as-is from `model/train.py` -- same stratified split, same
augmentation-boost logic for underrepresented classes, same EfficientNet-B0
transfer learning setup, same class-weighted loss, same per-class
precision/recall/F1 reporting. Only the environment (paths, no argparse)
changed.

In [ ]:
import random
import time
from collections import Counter, defaultdict

import numpy as np
import torch.nn as nn
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
VAL_FRACTION = 0.2
RANDOM_SEED = 42

# Classes with fewer than this many training images get the heavier
# augmentation pipeline (see model/train.py for the same constant/reasoning).
AUGMENTATION_BOOST_THRESHOLD = 400


def build_sample_list(processed_dir):
    class_names = sorted(p.name for p in processed_dir.iterdir() if p.is_dir())
    samples = []
    for class_index, class_name in enumerate(class_names):
        class_dir = processed_dir / class_name
        for entry in sorted(class_dir.iterdir()):
            if entry.is_file() and entry.suffix.lower() in IMAGE_EXTENSIONS:
                samples.append((entry, class_index))
    return samples, class_names


def stratified_split(samples):
    paths = [s[0] for s in samples]
    labels = [s[1] for s in samples]
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        paths, labels, test_size=VAL_FRACTION, stratify=labels, random_state=RANDOM_SEED
    )
    return list(zip(train_paths, train_labels)), list(zip(val_paths, val_labels))


def verify_split_covers_every_class(train_samples, val_samples, class_names):
    train_counts = Counter(label for _, label in train_samples)
    val_counts = Counter(label for _, label in val_samples)
    missing_train = [name for i, name in enumerate(class_names) if train_counts.get(i, 0) == 0]
    missing_val = [name for i, name in enumerate(class_names) if val_counts.get(i, 0) == 0]
    if missing_train or missing_val:
        print("WARNING: stratified split left classes without data:")
        for name in missing_train:
            print(f"  - {name}: 0 training images")
        for name in missing_val:
            print(f"  - {name}: 0 validation images")
    else:
        print(f"OK: all {len(class_names)} classes have both training and validation images.")
    return train_counts, val_counts


def build_transforms():
    standard_train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    heavy_train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(35),
        transforms.ColorJitter(brightness=0.4, contrast=0.4),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    val_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return standard_train_transform, heavy_train_transform, val_transform


def classes_needing_augmentation_boost(train_counts, class_names):
    boosted = {
        class_index
        for class_index, name in enumerate(class_names)
        if train_counts.get(class_index, 0) < AUGMENTATION_BOOST_THRESHOLD
    }
    print(f"\nClasses receiving heavier augmentation (< {AUGMENTATION_BOOST_THRESHOLD} training images):")
    for class_index in sorted(boosted):
        print(f"  - {class_names[class_index]} ({train_counts[class_index]} training images)")
    return boosted


class PlantDiseaseDataset(Dataset):
    def __init__(self, samples, transform_by_label):
        self.samples = samples
        self.transform_by_label = transform_by_label

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        transform = self.transform_by_label[label]
        return transform(image), label


def build_model(num_classes):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def compute_class_weights(train_counts, num_classes):
    labels_array = np.concatenate([np.full(train_counts.get(i, 0), i) for i in range(num_classes)])
    weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_classes), y=labels_array)
    return torch.tensor(weights, dtype=torch.float32)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


def evaluate(model, loader, device, class_names):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    return classification_report(
        all_labels, all_preds, labels=list(range(len(class_names))),
        target_names=class_names, output_dict=True, zero_division=0,
    )


def print_per_class_report(report, class_names):
    print(f"\n{'Class':35s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>8s}")
    print("-" * 76)
    for name in class_names:
        m = report[name]
        print(f"{name:35s} {m['precision']:10.3f} {m['recall']:10.3f} {m['f1-score']:10.3f} {int(m['support']):8d}")
    print("-" * 76)
    print(f"Overall accuracy: {report['accuracy']:.3f}")
    print(f"Macro F1:         {report['macro avg']['f1-score']:.3f}")
    print(f"Weighted F1:      {report['weighted avg']['f1-score']:.3f}")


print("Training functions defined.")


## 7. Run training -- 15 epochs, checkpoint saved every epoch

Each checkpoint is saved to Drive as `efficientnet_b0_epoch<NN>.pt` and
includes that epoch's validation report, so:

- nothing is lost if the Colab runtime disconnects mid-training (Section 8
  can rebuild the summary table from whatever checkpoints made it to Drive)
- you can compare validation performance across epochs afterward and load
  whichever epoch actually generalized best, instead of assuming epoch 15 is
  automatically the best one.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2  # Drive-backed image reads benefit from a couple of parallel workers
EPOCHS = 15
LR = 1e-4

samples, class_names = build_sample_list(DATA_PROCESSED)
print(f"Found {len(samples)} images across {len(class_names)} classes")

train_samples, val_samples = stratified_split(samples)
train_counts, val_counts = verify_split_covers_every_class(train_samples, val_samples, class_names)
print(f"Train: {len(train_samples)} images, Val: {len(val_samples)} images")

boosted_labels = classes_needing_augmentation_boost(train_counts, class_names)
standard_transform, heavy_transform, val_transform = build_transforms()

train_transform_by_label = {
    i: (heavy_transform if i in boosted_labels else standard_transform) for i in range(len(class_names))
}
val_transform_by_label = {i: val_transform for i in range(len(class_names))}

train_dataset = PlantDiseaseDataset(train_samples, train_transform_by_label)
val_dataset = PlantDiseaseDataset(val_samples, val_transform_by_label)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = build_model(len(class_names)).to(device)
class_weights = compute_class_weights(train_counts, len(class_names)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

epoch_reports = {}  # epoch number -> classification_report dict (also used by Section 8)

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    elapsed_min = (time.time() - start) / 60
    print(f"\nEpoch {epoch}/{EPOCHS} -- train loss: {train_loss:.4f} -- {elapsed_min:.1f} min")

    report = evaluate(model, val_loader, device, class_names)
    print_per_class_report(report, class_names)
    epoch_reports[epoch] = report

    checkpoint_path = CHECKPOINT_DIR / f"efficientnet_b0_epoch{epoch:02d}.pt"
    torch.save(
        {"model_state_dict": model.state_dict(), "class_names": class_names, "epoch": epoch, "val_report": report},
        checkpoint_path,
    )
    print(f"Saved checkpoint: {checkpoint_path}")


## 8. Per-class F1 summary across all epochs

Rebuilds the table from checkpoints saved on Drive (not just the in-memory
`epoch_reports` from the cell above), so this still works even if training
was interrupted and you're re-running just this cell in a fresh session.
Use it to check whether classes like `Sugarcane_Red_Rot` or the Rice classes
are consistently weak across every epoch (a data problem) or improve as
training progresses (an epoch-count problem).

In [ ]:
import pandas as pd


def load_all_checkpoint_reports(checkpoint_dir):
    reports = {}
    for ckpt_path in sorted(checkpoint_dir.glob("efficientnet_b0_epoch*.pt")):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        if "epoch" in ckpt and "val_report" in ckpt:
            reports[ckpt["epoch"]] = ckpt["val_report"]
    return reports


all_reports = load_all_checkpoint_reports(CHECKPOINT_DIR)
if not all_reports:
    all_reports = epoch_reports  # fall back to this session's in-memory results

class_names_for_table = class_names if "class_names" in dir() else sorted(
    {name for report in all_reports.values() for name in report if name not in
     ("accuracy", "macro avg", "weighted avg")}
)

rows = []
for epoch in sorted(all_reports):
    report = all_reports[epoch]
    for class_name in class_names_for_table:
        rows.append({"epoch": epoch, "class": class_name, "f1": report[class_name]["f1-score"]})

f1_pivot = pd.DataFrame(rows).pivot(index="class", columns="epoch", values="f1")
print(f1_pivot.round(3).to_string())

print("\nMacro F1 per epoch:")
for epoch in sorted(all_reports):
    print(f"  epoch {epoch}: {all_reports[epoch]['macro avg']['f1-score']:.3f}")
